<a href="https://colab.research.google.com/github/sofiiamyshelova-prog/python_for_ds_tasks/blob/main/Copy_of_HW_2_4_kNN_%D0%9A%D1%80%D0%BE%D1%81%D0%B2%D0%B0%D0%BB%D1%96%D0%B4%D0%B0%D1%86%D1%96%D1%8F_%D1%96_%D1%82%D1%8E%D0%BD%D0%B8%D0%BD%D0%B3_%D0%B3%D1%96%D0%BF%D0%B5%D1%80%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D1%96%D0%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В цьому домашньому завданні ми знову працюємо з даними з нашого змагання ["Bank Customer Churn Prediction (DLU Course)"](https://www.kaggle.com/t/7c080c5d8ec64364a93cf4e8f880b6a0).

Тут ми побудуємо рішення задачі класифікації з використанням kNearestNeighboors, знайдемо оптимальні гіперпараметри для цього методу і зробимо базові ансамблі. Це дасть змогу порівняти перформанс моделі з попередніми вивченими методами.

0. Зчитайте дані `train.csv` та зробіть препроцесинг використовуючи написаний Вами скрипт `process_bank_churn.py` так, аби в результаті отримати дані в розбитті X_train, train_targets, X_val, val_targets для експериментів.

  Якщо Вам не вдалось реалізувати в завданні `2.3. Дерева прийняття рішень` скрипт `process_bank_churn.py` - можна скористатись готовим скриптом з запропонованого рішення того завдання.

In [2]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

train = pd.read_csv('/content/drive/MyDrive/ML/data/train.csv')


from process_bank_churn import preprocess_data
data = preprocess_data(train, scale_numeric=False)

X_train = data["train_X"]
train_targets = data["train_y"]

X_val = data["val_X"]
val_targets = data["val_y"]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1. Навчіть на цих даних класифікатор kNN з параметрами за замовченням і виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах. Зробіть заключення про отриману модель: вона хороша/погана, чи є high bias/high variance?

In [3]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score

# Навчання моделі
knn = KNeighborsClassifier()
knn.fit(X_train, train_targets)

# Передбачення ймовірностей
train_preds = knn.predict_proba(X_train)[:, 1]
val_preds = knn.predict_proba(X_val)[:, 1]

# AUROC
train_auc = roc_auc_score(train_targets, train_preds)
val_auc = roc_auc_score(val_targets, val_preds)

print(f"Train ROC AUC: {train_auc:.4f}")
print(f"Validation ROC AUC: {val_auc:.4f}")

Train ROC AUC: 0.8238
Validation ROC AUC: 0.5734


Train невисокий, але норм.

Val дуже низький. Ледь вищий за вгадування.

Модель перенавчена high variance




2. Використовуючи `GridSearchCV` знайдіть оптимальне значення параметра `n_neighbors` для класифікатора `kNN`. Псотавте крос валідацію на 5 фолдів.

  Після успішного завершення пошуку оптимального гіперпараметра
    - виведіть найкраще значення параметра
    - збережіть в окрему змінну `knn_best` найкращу модель, знайдену з `GridSearchCV`
    - оцініть якість передбачень  `knn_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пукнтом (2) цього завдання? Чи є вона краще за дерево прийняття рішень з попереднього ДЗ?

In [4]:
from sklearn.model_selection import GridSearchCV


# Сітка параметрів
param_grid = {
    "n_neighbors": list(range(1, 21))
}

# Grid Search з 5-fold CV
grid_search = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc"
)

# Навчання
grid_search.fit(X_train, train_targets)

print("Best n_neighbors:", grid_search.best_params_["n_neighbors"])
print("Best CV ROC AUC:", grid_search.best_score_)

knn_best = grid_search.best_estimator_

train_preds = knn_best.predict_proba(X_train)[:, 1]
val_preds = knn_best.predict_proba(X_val)[:, 1]

train_auc = roc_auc_score(train_targets, train_preds)
val_auc = roc_auc_score(val_targets, val_preds)

print(f"Train ROC AUC: {train_auc:.4f}")
print(f"Validation ROC AUC: {val_auc:.4f}")

Best n_neighbors: 20
Best CV ROC AUC: 0.5807283547803754
Train ROC AUC: 0.6968
Validation ROC AUC: 0.6107


Train ROC AUC зменшився з 0.8238 до 0.6968.

Validation ROC AUC зріс з 0.5734 до 0.6107.

Модель стала менш перенавченою, але все ще погана. Гірша за дерево.

3. Виконайте пошук оптимальних гіперпараметрів для `DecisionTreeClassifier` з `GridSearchCV` за сіткою параметрів
  - `max_depth` від 1 до 20 з кроком 2
  - `max_leaf_nodes` від 2 до 10 з кроком 1

  Обовʼязково при цьому ініціюйте модель з фіксацією `random_state`.

  Поставте кросвалідацію на 3 фолди, `scoring='roc_auc'`, та виміряйте, скільки часу потребує пошук оптимальних гіперпараметрів.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення параметра
    - збережіть в окрему змінну `dt_best` найкращу модель, знайдену з `GridSearchCV`
    - оцініть якість передбачень  `dt_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи ця модель краща за ту, що ви знайшли вручну?

In [5]:
from sklearn.tree import DecisionTreeClassifier
import time

dt = DecisionTreeClassifier(random_state=42)

# Створюємо сітку параметрів
param_grid = {
    "max_depth": range(1, 21, 2),
    "max_leaf_nodes": range(2, 11)
}


# Створюємо GridSearchCV
grid_search_dt = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    cv=3,
    scoring="roc_auc"
)


# Вимірюємо час пошуку
start_time = time.time()

grid_search_dt.fit(
    X_train,
    train_targets
)

end_time = time.time()

search_time = end_time - start_time

# Виводимо результати пошуку
print("Best parameters:", grid_search_dt.best_params_)
print(f"Best CV ROC AUC: {grid_search_dt.best_score_:.4f}")
print(f"GridSearch time: {search_time:.2f} seconds")

# Зберігаємо найкращу модель
dt_best = grid_search_dt.best_estimator_

# Робимо передбачення
train_preds = dt_best.predict_proba(X_train)[:, 1]
val_preds = dt_best.predict_proba(X_val)[:, 1]

# Обчислюємо AUROC
train_auc = roc_auc_score(
    train_targets,
    train_preds
)

val_auc = roc_auc_score(
    val_targets,
    val_preds
)

# Виводимо результати
print(f"Train ROC AUC: {train_auc:.4f}")
print(f"Validation ROC AUC: {val_auc:.4f}")

Best parameters: {'max_depth': 5, 'max_leaf_nodes': 10}
Best CV ROC AUC: 0.9014
GridSearch time: 6.48 seconds
Train ROC AUC: 0.9015
Validation ROC AUC: 0.9002


модель якісна

Train ROC AUC: 0.9015

Validation ROC AUC: 0.9002

обидва ROC AUC високі. модель добре навчилась



4. Виконайте пошук оптимальних гіперпараметрів для `DecisionTreeClassifier` з `RandomizedSearchCV` за заданою сіткою параметрів і кількість ітерацій 40.

  Поставте кросвалідацію на 3 фолди, `scoring='roc_auc'`, зафіксуйте `random_seed` процедури крос валідації та виміряйте, скільки часу потребує пошук оптимальних гіперпараметрів.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення параметра
    - збережіть в окрему змінну `dt_random_search_best` найкращу модель, знайдену з `RandomizedSearchCV`
    - оцініть якість передбачень  `dt_random_search_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи ця модель краща за ту, що ви знайшли з `GridSearch`?
    - проаналізуйте параметри `dt_random_search_best` і порівняйте з параметрами `dt_best` - яку бачите відмінність? Ця вправа потрібна аби зрозуміти, як різні налаштування `DecisionTreeClassifier` впливають на якість моделі.

In [12]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

params_dt = {
    'criterion': ['gini', 'entropy'],
    'splitter': ['best', 'random'],
    'max_depth': np.arange(1, 20),
    'max_leaf_nodes': np.arange(2, 20),
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': [None, 'sqrt', 'log2']
}


# Randomized Search
random_search = RandomizedSearchCV(
    estimator=dt,
    param_distributions=params_dt,
    n_iter=40,
    scoring="roc_auc",
    cv=cv,
    random_state=42
)

# Вимірювання часу
start_time = time.time()

random_search.fit(X_train, train_targets)

end_time = time.time()

print(f"Search time: {end_time - start_time:.2f} seconds")
print("Best parameters:", random_search.best_params_)
print(f"Best CV ROC AUC: {random_search.best_score_:.4f}")

dt_random_search_best = random_search.best_estimator_

train_preds = dt_random_search_best.predict_proba(X_train)[:, 1]
val_preds = dt_random_search_best.predict_proba(X_val)[:, 1]

train_auc = roc_auc_score(train_targets, train_preds)
val_auc = roc_auc_score(val_targets, val_preds)

print(f"Train ROC AUC: {train_auc:.4f}")
print(f"Validation ROC AUC: {val_auc:.4f}")

#порівняння параметрів
print("GridSearch parameters:")
print(dt_best.get_params())

print("\nRandomizedSearch parameters:")
print(dt_random_search_best.get_params())

Search time: 2.93 seconds
Best parameters: {'splitter': 'best', 'min_samples_split': 20, 'min_samples_leaf': 2, 'max_leaf_nodes': np.int64(14), 'max_features': None, 'max_depth': np.int64(16), 'criterion': 'entropy'}
Best CV ROC AUC: 0.9103
Train ROC AUC: 0.9169
Validation ROC AUC: 0.9166
GridSearch parameters:
{'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 5, 'max_features': None, 'max_leaf_nodes': 10, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'random_state': 42, 'splitter': 'best'}

RandomizedSearch parameters:
{'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'entropy', 'max_depth': np.int64(16), 'max_features': None, 'max_leaf_nodes': np.int64(14), 'min_impurity_decrease': 0.0, 'min_samples_leaf': 2, 'min_samples_split': 20, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'random_state': 42, 'splitter': 'best'}


5. Якщо у Вас вийшла метрика `AUROC` в цій серії експериментів - зробіть ще один `submission` на Kaggle і додайте код для цього і скріншот скора на публічному лідерборді нижче.

  Сподіваюсь на цьому етапі ви вже відчули себе справжнім дослідником 😉

не буду додавати новий submission на Kaggle, бо результат гірший, ніж поточний.